# Day 1 — Spotify Streaming History Exploration

Goal: load my Spotify export, sanity-check the data, and produce the cleaned Parquet files that day 2+ will consume.

**Outputs we want by end of notebook:**
- `data/processed/all_streams.parquet` — every stream event
- `data/processed/real_plays.parquet` — plays >= 30s only
- `data/processed/plays_by_artist.parquet` — recommender input (artist-level)
- `data/processed/plays_by_track.parquet` — for analytics
- `data/processed/daily_listening.parquet` — for the dashboard

**Sanity questions to answer before moving on:**
1. How many plays total? Date range?
2. What's my top-N artists list? Does it match my intuition?
3. Skip rate — what fraction of streams are < 30s?
4. Any weird data quality issues (duplicates, missing fields, podcasts mixed in)?

In [ ]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

# Make the src package importable from the notebook
sys.path.insert(0, str(Path.cwd().parent / 'src'))

import pandas as pd
import plotly.express as px
from spotify_recs import loader

## 1. Load raw streams

In [ ]:
streams = loader.load_streams('../data/raw')
print(f'Total events: {len(streams):,}')
print(f'Date range:   {streams["ts"].min()} → {streams["ts"].max()}')
print(f'Span:         {(streams["ts"].max() - streams["ts"].min()).days} days')
streams.head()

In [ ]:
# Field coverage — what fraction of rows have each optional field?
# (Tells us which export format(s) we have)
streams.notna().mean().sort_values(ascending=False).round(3)

## 2. Skip rate & play threshold

In [ ]:
# Distribution of play durations. The 30s threshold should be a natural cutoff.
fig = px.histogram(
    streams[streams['ms_played'] < 300_000],  # zoom to first 5 min
    x='ms_played',
    nbins=60,
    title='Distribution of play durations (capped at 5 min)',
)
fig.add_vline(x=30_000, line_dash='dash', annotation_text='30s threshold')
fig.show()

In [ ]:
skip_rate = (streams['ms_played'] < 30_000).mean()
print(f'Skip rate (< 30s): {skip_rate:.1%}')
print(f'Plays kept: {(streams["ms_played"] >= 30_000).sum():,} / {len(streams):,}')

## 3. Top artists — gut check

In [ ]:
plays = loader.filter_real_plays(streams)
by_artist = loader.aggregate_by_artist(plays)
by_artist.head(20)

In [ ]:
# Quick look at the long tail — most artists in your library are heard once or twice
fig = px.histogram(
    by_artist, x='play_count', log_y=True,
    title=f'Play count distribution per artist (n={len(by_artist):,} artists)',
    nbins=50,
)
fig.show()

print(f'Artists played only once: {(by_artist["play_count"] == 1).sum():,}')
print(f'Artists played 10+ times: {(by_artist["play_count"] >= 10).sum():,}')

## 4. Listening over time

In [ ]:
daily = loader.daily_listening(plays)
fig = px.line(daily, x='date', y='minutes', title='Daily listening (minutes)')
fig.update_traces(line_width=1)
fig.show()

In [ ]:
# Hour-of-day pattern — when are you listening?
plays_local = plays.copy()
plays_local['hour'] = plays_local['ts'].dt.tz_convert('America/New_York').dt.hour
hourly = plays_local.groupby('hour').size().reset_index(name='plays')
fig = px.bar(hourly, x='hour', y='plays', title='Plays by hour of day (local time)')
fig.show()

## 5. Data quality checks

In [ ]:
# Are there exact-duplicate plays? (same artist+track+timestamp)
dupes = streams.duplicated(subset=['ts', 'artist_name', 'track_name']).sum()
print(f'Exact duplicate rows: {dupes:,}')

# Any artists that look like podcasts/ambient junk that snuck through?
# (Long tail of single plays often catches these)
print('\nArtists with exactly 1 play (sample):')
by_artist[by_artist['play_count'] == 1]['artist_name'].sample(
    min(15, (by_artist['play_count'] == 1).sum()), random_state=0
).tolist()

## 6. Save processed files

Once the above looks reasonable, write everything out as Parquet.

In [ ]:
paths = loader.save_processed(streams, out_dir='../data/processed')
paths

---

## Day 1 checklist

- [ ] All JSON files loaded without errors
- [ ] Date range matches expectation (~1 year for Account Data)
- [ ] Top artists list passes the gut check
- [ ] Skip rate looks reasonable (typically 20–40%)
- [ ] No obvious data quality issues
- [ ] Parquet files written to `data/processed/`

Day 2 picks up from `plays_by_artist.parquet`.